# HWD — Hierarchical Wavelet Diffusion
**Partial Imputation untuk Official Statistics (Data IMK BPS)**

Jalankan cell **secara berurutan**. Jangan skip.

---
### Catatan Preprocessing
- **Normalisasi**: Z-score per kolom, hanya dari nilai observed (skip -200/NaN)
- **Missing placeholder**: nilai `-200` untuk KDD/Guangzhou, `NaN` untuk IMK
- **Reshaping**: dilakukan di `dataset.py` — sliding window tanpa overlap (`stride=seq_len`)
- **Split**: 70% train / 30% test, dilakukan setelah reshape
- **Mask file**: dibuat oleh `generate_missing.py`, dijalankan sekali sebelum training
- **Epoch**: 200 untuk KDD/Guangzhou/Physio (identik FGTI), 300 untuk IMK
- **Checkpoint**: disimpan otomatis ke Google Drive setiap 50 epoch dan di akhir training

## 1. Setup & Install

In [ ]:
!pip install PyWavelets properscoring --quiet
print('Install selesai')

In [ ]:
import os, sys, torch, numpy as np

from google.colab import drive
drive.mount('/content/drive')

# Sesuaikan path ini
PROJECT_DIR = '/content/drive/MyDrive/Skripsi/HWD/Code'
CKPT_DIR    = '/content/drive/MyDrive/Skripsi/HWD/Checkpoints'
DATASET     = 'kdd'   # ganti: 'guangzhou' | 'physio' | 'imk'

os.makedirs(CKPT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

print(f'Working dir : {os.getcwd()}')
print(f'Checkpoint  : {CKPT_DIR}')
print(f'Files       : {sorted(os.listdir("."))}')

## 2. Preprocessing
Jalankan **sekali saja** — skip jika file `*_norm.csv` sudah ada.

In [ ]:
import pandas as pd

def preprocess_zscore(raw_path, out_path, missing_val=-200):
    """Z-score per kolom, skip nilai missing_val. Identik FGTI."""
    if os.path.exists(out_path):
        print(f'Skip — {out_path} sudah ada'); return
    data = pd.read_csv(raw_path, header=0).to_numpy().astype(float)
    for j in range(data.shape[1]):
        obs = data[data[:, j] != missing_val, j]
        obs = obs[~np.isnan(obs)]
        if len(obs) == 0: continue
        m, s = obs.mean(), obs.std() + 1e-8
        mask = (data[:, j] != missing_val) & ~np.isnan(data[:, j])
        data[mask, j] = (data[mask, j] - m) / s
    np.savetxt(out_path, data, delimiter=',', fmt='%6f')
    print(f'Saved: {out_path}  shape={data.shape}')

# KDD perlu reshape tambahan (9 stasiun)
def preprocess_kdd(raw='Data/KDD.csv', out='Data/KDD_norm.csv'):
    if os.path.exists(out):
        print(f'Skip — {out} sudah ada'); return
    data = pd.read_csv(raw, header=0).to_numpy()
    lst  = [data[:, i*13+2:(i+1)*13] for i in range(9)]
    data = np.stack(lst, axis=1).reshape(data.shape[0], -1).astype(float)
    for j in range(data.shape[1]):
        obs = data[data[:, j] != -200, j]
        if len(obs) == 0: continue
        m, s = obs.mean(), obs.std() + 1e-8
        data[data[:, j] != -200, j] = (data[data[:, j] != -200, j] - m) / s
    np.savetxt(out, data, delimiter=',', fmt='%6f')
    print(f'Saved: {out}  shape={data.shape}')

if DATASET == 'kdd':
    preprocess_kdd()
elif DATASET == 'imk':
    preprocess_zscore('Data/IMK_raw.csv', 'Data/IMK_norm.csv', missing_val=np.nan)
else:
    print(f'{DATASET}: tidak perlu preprocessing tambahan')

## 3. Generate Missing Masks
Jalankan **sekali saja**.

In [ ]:
import generate_missing as gm

RATES = [0.1, 0.2, 0.3, 0.4]
SEEDS = [3407]

if DATASET in ('kdd', 'guangzhou', 'physio'):
    gm.generate_all_masks(DATASET, ['mcar','mar','mnar'], RATES, SEEDS)
else:
    gm.generate_all_masks('imk', ['mcar','block'], RATES, SEEDS,
                          data_path='Data/IMK_norm.csv', block_size=2)

## 4. Konfigurasi

In [ ]:
import train_OutlierRemoval

PARAMS = {
    'kdd'      : {'seq_len':48,'enc_in':99, 'c_out':99},
    'guangzhou': {'seq_len':48,'enc_in':214,'c_out':214},
    'physio'   : {'seq_len':48,'enc_in':37, 'c_out':37},
    'imk'      : {'seq_len':12,'enc_in':0,  'c_out':0},
}
MISSING_RATE = 0.1   # ganti: 0.1 | 0.2 | 0.3 | 0.4
p = PARAMS[DATASET]

configs = train_OutlierRemoval.get_config({
    'dataset'          : DATASET,
    'missing_rate'     : MISSING_RATE,
    'seed'             : 3407,
    'seq_len'          : p['seq_len'],
    'enc_in'           : p['enc_in'],
    'c_out'            : p['c_out'],
    # Model
    'd_model'          : 128,
    'e_layers'         : 4,
    'nheads'           : 8,
    'channel'          : 128,
    'proj_t'           : 128,
    'residual_layers'  : 4,
    'timeemb'          : 128,
    'featureemb'       : 16,
    # Diffusion
    'diffusion_step_num': 50,
    'schedule'          : 'quad',
    'beta_start'        : 0.0001,
    'beta_end'          : 0.2,
    # Epoch: 200 untuk benchmark, 300 untuk IMK
    'epoch_diff'        : 200 if DATASET != 'imk' else 300,
    'learning_rate_diff': 1e-3,
    # SSL
    'mask_ratio_ssl'   : 0.2,
    'avg_mask_len_ssl' : 3,
    # Wavelet HWD
    'wavelet'          : 'db4',
    'levels'           : 3,
    # Runtime
    'batch'            : 16,
    'device'           : 'cuda' if torch.cuda.is_available() else 'cpu',
    'n_samples'        : 100,
    'save_dir'         : CKPT_DIR,
    # IMK
    'data_path'        : 'Data/IMK_norm.csv',
    'col_names_path'   : '',
    'exp_mask_path'    : f'Data/mask/imk/imkblock2_{MISSING_RATE}_3407.csv',
})

print(f'Dataset : {configs.dataset}  |  Device : {configs.device}')
print(f'enc_in  : {configs.enc_in}  |  Epochs : {configs.epoch_diff}')
print(f'Wavelet : {configs.wavelet}  levels={configs.levels}')
print(f'Ckpt dir: {configs.save_dir}')

## 5. Training
> Checkpoint disimpan otomatis ke Drive setiap 50 epoch dan di akhir.
> Kalau Colab disconnect, lanjutkan dari cell **Resume** di bawah.

In [ ]:
np.random.seed(configs.seed)
torch.manual_seed(configs.seed)
if configs.device == 'cuda': torch.cuda.manual_seed(configs.seed)

model = train_OutlierRemoval.diffusion_train(configs)
print('Training selesai.')

## 5b. Resume dari Checkpoint (jika disconnect)

In [ ]:
from models import main_model

CKPT_PATH = f"{CKPT_DIR}/hwd_{configs.dataset}_mr{configs.missing_rate}.pt"
model = main_model.HWD(configs).to(configs.device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=configs.device))
model.eval()
print(f'Loaded: {CKPT_PATH}')

## 6. Evaluasi — HWD vs Mean Imputation
Mean Imputation sebagai lower bound.
Hasil CSDI/FGTI/NAOMI diambil dari tabel paper masing-masing.

In [ ]:
# HWD
hwd_res = train_OutlierRemoval.diffusion_test(configs, model)
print(f"HWD  -> MAE={hwd_res['MAE']:.4f}  RMSE={hwd_res['RMSE']:.4f}  CRPS={hwd_res['CRPS']:.4f}")

In [ ]:
import dataset as ds_module

train_loader, test_loader = ds_module.get_dataset(configs)

# Hitung mean kolom dari training set (hanya observed)
sum_v, sum_c = None, None
for obs_d, obs_m, _, _ in train_loader:
    v = (obs_d * obs_m).sum(dim=(0,1))
    c = obs_m.sum(dim=(0,1))
    sum_v = v if sum_v is None else sum_v + v
    sum_c = c if sum_c is None else sum_c + c
col_means = sum_v / (sum_c + 1e-8)   # [K]

mae_t, rmse_t, pts = 0.0, 0.0, 0
for obs_d, obs_m, _, gt_m in test_loader:
    B, L, K  = obs_d.shape
    ev       = (gt_m - obs_m).clamp(0,1)
    mean_imp = col_means.unsqueeze(0).unsqueeze(0).expand(B,L,K)
    imp      = obs_d * obs_m + mean_imp * (1 - obs_m)
    idx      = torch.where(ev==1)
    diff     = (imp[idx] - obs_d[idx]).abs()
    mae_t   += diff.sum().item()
    rmse_t  += (diff**2).sum().item()
    pts     += len(idx[0])

mean_MAE  = mae_t  / (pts+1e-8)
mean_RMSE = (rmse_t / (pts+1e-8))**0.5
print(f"Mean -> MAE={mean_MAE:.4f}  RMSE={mean_RMSE:.4f}  CRPS=N/A")

In [ ]:
print(f'\n{chr(9552)*52}')
print(f'  Dataset : {configs.dataset.upper()}   Missing rate : {configs.missing_rate}')
print(f'{chr(9472)*52}')
print(f'  {"Model":<20} {"MAE":>7} {"RMSE":>7} {"CRPS":>7}')
print(f'{chr(9472)*52}')
print(f'  {"Mean Imputation":<20} {mean_MAE:>7.4f} {mean_RMSE:>7.4f} {"N/A":>7}')
print(f'  {"HWD (ours)":<20} {hwd_res["MAE"]:>7.4f} {hwd_res["RMSE"]:>7.4f} {hwd_res["CRPS"]:>7.4f}')
print(f'{chr(9472)*52}')
print(f'  Hasil CSDI/FGTI/NAOMI -> lihat tabel paper masing-masing')
print(f'{chr(9552)*52}')

## 7. Visualisasi Imputasi

In [ ]:
import matplotlib.pyplot as plt

batch = next(iter(test_loader))
obs_d, obs_m, obs_tp, gt_m = batch

model.eval()
with torch.no_grad():
    out = model.evaluate(obs_d, obs_m, obs_tp, gt_m, n_samples=50)

imp_samps, c_target, eval_pts, _, _ = out
imp_med = imp_samps.median(dim=1).values   # [B, K, L]

B_idx, feat_idx = 0, 0
truth   = c_target[B_idx, feat_idx, :].cpu().numpy()
imputed = imp_med[B_idx, feat_idx, :].cpu().numpy()
ev_mask = eval_pts[B_idx, feat_idx, :].cpu().numpy()
samps   = imp_samps[B_idx, :, feat_idx, :].cpu().numpy()

T = len(truth)
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(range(T), truth,   'k-',  lw=1.5, label='Ground truth')
ax.plot(range(T), imputed, '--',  lw=1.5, color='#4f9eff', label='HWD (median)')
lo = np.percentile(samps, 5, axis=0)
hi = np.percentile(samps, 95, axis=0)
ax.fill_between(range(T), lo, hi, alpha=0.2, color='#4f9eff', label='5-95th pct')
miss_idx = [i for i in range(T) if ev_mask[i]==1]
if miss_idx:
    ax.scatter(miss_idx, truth[miss_idx], c='red', s=25, zorder=5, label='Missing (Omega)')
ax.set_title(f'{configs.dataset.upper()} | Feature {feat_idx} | mr={configs.missing_rate}')
ax.set_xlabel('Timestep')
ax.legend(fontsize=8)
plt.tight_layout()
out_png = f"{CKPT_DIR}/imputation_{configs.dataset}_mr{configs.missing_rate}.png"
plt.savefig(out_png, dpi=150)
plt.show()
print(f'Plot disimpan: {out_png}')

## 8. Incremental Learning (opsional)
Hanya untuk IMK — saat data periode baru tersedia.

In [ ]:
import incremental

BASE_CKPT    = f"{CKPT_DIR}/hwd_{configs.dataset}_mr{configs.missing_rate}.pt"
NEW_CSV_PATH = ''   # isi path data baru

configs_new = train_OutlierRemoval.get_config({**vars(configs), 'data_path': NEW_CSV_PATH})
new_train_loader, new_test_loader = ds_module.get_dataset(configs_new)

model_base = main_model.HWD(configs).to(configs.device)
model_base.load_state_dict(torch.load(BASE_CKPT, map_location=configs.device))

model_incr = incremental.incremental_finetune(
    model_base, new_train_loader,
    freeze_ratio=0.8, lr=1e-5, epochs=50
)
incr_ckpt = BASE_CKPT.replace('.pt','_incr.pt')
torch.save(model_incr.state_dict(), incr_ckpt)

_, old_test = ds_module.get_dataset(configs)
result = incremental.run_forgetting_check(
    BASE_CKPT, incr_ckpt, configs, old_test, new_test_loader
)
print(f"Forgetting  : {result['forgetting_pct']:+.2f}%  (target < 5%)")
print(f"Improvement : {result['improvement_pct']:+.2f}%  (target > 0%)")